In [1]:
import numpy as np
import math

# Практические задания

#### 1. Своё цепное правило. Реализуйте функцию chain(funcs, derivs, x), которая последовательно применяет функции и возвращает (value, derivative). Проверьте на y = sin(exp(x²)).

In [13]:
def chain(funcs, derivs, x):
    """
    Вычисляет значение композиции функций и её производную
    по цепному правилу (прямой и обратный проход).
    """
    # Прямой проход (Forward pass)
    values = [x]   # Массив для хранения всех промежуточных значений
    current_value = x

    for i in range(len(funcs)):
        current_value = funcs[i](current_value)
        values.append(current_value)

    # Обратный проход (Backward pass)
    derivative = 1   # Начальное значение градиента

    for i in range(len(funcs) -1, -1, -1):   # Идём с конца
        input_val = values[i]

        # Считаем локальную производную
        local_derivative = derivs[i](input_val)

        # Накапливаем градиент
        derivative = derivative * local_derivative

    return current_value, derivative   # Возвращаем результат

# Функции в порядке применения
funcs = [lambda v: 2 * v + 1, lambda v: v ** 2, np.sin,]

# Их производные
derivs = [lambda v: 2, lambda v: 2 * v, np.cos,]

# Вычисление значения и производной в точке x = 0.5
x = 0.5
value, deriv = chain(funcs, derivs, x)

print(f"F({x})  {value:.6f}")
print(f"F'({x}) {deriv:.6f}")

# Проверка численной производной
def F(x):
    v = x
    for f in funcs:
        v = f(v)
    return v

h = 1e-6
num_deriv = (F(x + h) - F(x - h)) / (2 * h)
print(f"Численная производная: {num_deriv:.6f}")
print(f"Разница: {abs(deriv - num_deriv):.2e}")

assert abs(deriv - num_deriv) < 1e-5, "Производная вычислена неверно!"
print("Цепное правило работает!")

F(0.5)  -0.756802
F'(0.5) -5.229149
Численная производная: -5.229149
Разница: 2.59e-10
Цепное правило работает!


#### 2. Backprop для одного нейрона. Реализуйте forward и backward вручную для a = σ(wx + b), L = (a-y)². Сделайте 1000 шагов GD на одной точке (x=2, y=1). Покажите, что a → 1.

In [6]:
def sigmoid(z):
    return 1 / (1 + math.exp(-z))

# Гиперпараметры
learning_rate = 1.0   # Скорость обучения
x = 2                 # Вход
y = 1                 # Целевое значение
steps = 1000          # Количество шагов GD

# Инициализация параметров небольшими случайными числами
w = 0.5
b = 0.1

print(f"Начальные параметры: w={w:.4f}, b={b:.4f}")
print(f"Вход: x={x}, цель: y={y}")
print("=" * 60)

# Обучение методом градиентного спуска
for step in range(steps):
    # Прямой проход (Forward pass)
    z = w * x + b      # Линейная комбинация
    a = sigmoid(z)     # Активация a
    L = (a - y) ** 2   # Функция потерь
        
    # Обратный проход (Backward pass)
    dL_da = 2 * (a - y)     # Производная L по активации
    da_dz = a * (1 - a)     # Производная сигмоиды
    dL_dz = dL_da * da_dz   # Производная L по z (цепное правило)
        
    # Градиенты по параметрам
    dL_dw = dL_dz * x       
    dL_db = dL_dz * 1

    # Обновление параметров
    w = w - learning_rate * dL_dw
    b = b - learning_rate * dL_db

    # Вывод прогресса каждые 100 шагов
    if step % 100 == 0 or step == steps - 1:
        print(f"Шаг {step:4d}: a = {a:.6f}, L = {L:.6f}, "
              f"w = {w:.4f}, b = {b:.4f}, z = {z:.4f}")

# Финальная проверка
print("=" * 60)
print(f"Итоговые параметры: w = {w:.4f}, b = {b:.4f}")
z_final = w * x + b
a_final = sigmoid(z_final)
print(f"Итоговый выход нейрона: a = {a_final:.6f}")
print(f"Целевое назначение:     y = {y}")
print(f"Финальная ошибка L:     {(a_final - y) ** 2:.2e}")

assert a_final > 0.99, f"Нейрон не сходится к 1! a = {a_final}"
print("\nУспех! Нейрон научился выдавать а стремится к 1")

Начальные параметры: w=0.5000, b=0.1000
Вход: x=2, цель: y=1
Шаг    0: a = 0.750260, L = 0.062370, w = 0.6872, b = 0.1936, z = 1.1000
Шаг  100: a = 0.976995, L = 0.000529, w = 1.5616, b = 0.6308, z = 3.7488
Шаг  200: a = 0.983823, L = 0.000262, w = 1.7042, b = 0.7021, z = 4.1078
Шаг  300: a = 0.986832, L = 0.000173, w = 1.7874, b = 0.7437, z = 4.3167
Шаг  400: a = 0.988620, L = 0.000130, w = 1.8463, b = 0.7731, z = 4.4644
Шаг  500: a = 0.989837, L = 0.000103, w = 1.8919, b = 0.7960, z = 4.5788
Шаг  600: a = 0.990733, L = 0.000086, w = 1.9291, b = 0.8146, z = 4.6720
Шаг  700: a = 0.991428, L = 0.000073, w = 1.9606, b = 0.8303, z = 4.7507
Шаг  800: a = 0.991988, L = 0.000064, w = 1.9878, b = 0.8439, z = 4.8188
Шаг  900: a = 0.992451, L = 0.000057, w = 2.0117, b = 0.8559, z = 4.8788
Шаг  999: a = 0.992839, L = 0.000051, w = 2.0330, b = 0.8665, z = 4.9319
Итоговые параметры: w = 2.0330, b = 0.8665
Итоговый выход нейрона: a = 0.992843
Целевое назначение:     y = 1
Финальная ошибка L:     5.

#### 3. Микро‑autograd. Напишите класс Value с операциями +, *, tanh, который строит граф и метод backward() идёт по нему. Аналог micrograd Карпатого.

In [10]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = float(data)         # Значение
        self.grad = 0.0                 # Градиент dL/d(self)
        self._backward = lambda: None   # Локальная функция обратного прохода
        self._prev = set(_children)     # Родители в графе
        self._op = _op                  # Операция (для отладки)

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad

        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out
        
    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')

        def _backward():
            self.grad += (1 - t ** 2) * out.grad

        out._backward = _backward
        return out

    def backward(self):
        # Топологическая сортировка графа (дети после родителей)
        topo = []
        visited = set()

        def build_topo(node):
            if node not in visited:
                visited.add(node)
                for parent in node._prev:
                    build_topo(parent)
                topo.append(node)

        build_topo(self)

        self.grad = 1.0   # Градиент корня: dL/dL = 1

        # Обратный проход в ОБРАТНОМ топологическом порядке
        for node in reversed(topo):
            node._backward()

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

#### Пример использования + проверка градиентов:

In [11]:
# Нейрон: a = tanh(w*x + b), L = (a - 1)²
x = Value(2.0)
w = Value(0.5)
b = Value(0.1)

# Прямой проход
z = w * x + b
a = z.tanh()
diff = a + (-1.0)
L = diff * diff

# Обратный проход
L.backward()

print(f"a = {a.data:.6f}, L = {L.data:.6f}")
print(f"w.grad = {w.grad:.6f}, b.grad = {b.grad:.6f}")

# Проверка численным градиентом
def loss_fn(w_val, b_val, x_val=2.0, y=1.0):
    return(math.tanh(w_val * x_val + b_val) - y) ** 2

h = 1e-6
dw_num = (loss_fn(w.data + h, b.data) - loss_fn(w.data - h, b.data)) / (2 * h)
db_num = (loss_fn(w.data, b.data + h) - loss_fn(w.data, b.data - h)) / (2 * h)

print(f"Численный dw_num = {dw_num:.6f}, db_num = {db_num:.6f}")
assert abs(w.grad - dw_num) < 1e-5 and abs(b.grad - db_num) < 1e-5
print("Градиенты верны")

a = 0.800499, L = 0.039801
w.grad = -0.286644, b.grad = -0.143322
Численный dw_num = -0.286644, db_num = -0.143322
Градиенты верны


#### 4. Двухслойная сеть на MNIST. Реализуйте forward + backward вручную (без PyTorch) для архитектуры 784 → 64 → 10 с softmax + cross-entropy. Обучите до 90%+ accuracy.

#### 5. Проверка градиента численно. Для любой вашей сети сравните аналитический dL/dw с (L(w+h) - L(w-h))/(2h). Разница должна быть меньше 1e-5.

#### 6. Что будет, если ReLU. Возьмите сеть из задания 4. Замените ReLU на сигмоиду в скрытом слое. Что произойдёт с градиентами для глубокой сети (5+ слоёв)? Это называется vanishing gradient.

#### 7. Граф вычислений. Нарисуйте на бумаге граф для L = ||W₂ ReLU(W₁ x + b₁) + b₂ - y||². Подпишите все локальные градиенты.

#### 8. Тот же backprop в PyTorch. Повторите задание 2, но используя torch.tensor(..., requires_grad=True) и L.backward(). Сравните значения градиентов.